In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.utils import save_image


BATCH_SIZE = 16  
IMAGE_SIZE = 64
CHANNELS_IMG = 3
DRONE_CLASSES = 11
LATENT_DIM = 100
EMBED_SIZE = 100
FEATURES_G = 64
FEATURES_D = 64
FINE_TUNE_LR = 5e-5 
FT_EPOCHS = 30
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


class Generator(nn.Module):
    def __init__(self, num_classes, latent_dim, embed_size, img_channels, features_g):
        super(Generator, self).__init__()
        self.embed = nn.Embedding(num_classes, embed_size)
        self.net = nn.Sequential(
            nn.ConvTranspose2d(latent_dim + embed_size, features_g * 8, 4, 1, 0, bias= False),
            nn.BatchNorm2d(features_g * 8),
            nn.ReLU(True),

            nn.ConvTranspose2d(features_g * 8, features_g * 4, 4, 2, 1, bias= False),
            nn.BatchNorm2d(features_g * 4),
            nn.ReLU(True),
            
            nn.ConvTranspose2d(features_g * 4, features_g * 2, 4, 2, 1, bias= False),
            nn.BatchNorm2d(features_g * 2),
            nn.ReLU(True),
            
            nn.ConvTranspose2d(features_g * 2, features_g, 4, 2, 1, bias= False),
            nn.BatchNorm2d(features_g),
            nn.ReLU(True),
            
            nn.ConvTranspose2d(features_g, img_channels, 4, 2, 1, bias= False),
            nn.Tanh()
        )

    def forward(self, noise, labels):
        embedding = self.embed(labels).unsqueeze(2).unsqueeze(3) # [N, embed_size, 1, 1]
        x = torch.cat([noise, embedding], dim= 1)
        return self.net(x)


class Discriminator(nn.Module):
    def __init__(self, num_classes, img_channels, features_d, img_size=  64):
        super(Discriminator, self).__init__()
        self.img_size = img_size
        self.embed = nn.Embedding(num_classes, img_size * img_size)
        self.net = nn.Sequential(
            nn.Conv2d(img_channels + 1, features_d, 4, 2, 1, bias= False),
            nn.LeakyReLU(0.2, inplace= True),
            nn.Conv2d(features_d, features_d * 2, 4, 2, 1, bias= False),
            nn.BatchNorm2d(features_d * 2),
            nn.LeakyReLU(0.2, inplace= True),
            nn.Conv2d(features_d * 2, features_d * 4, 4, 2, 1, bias= False),
            nn.BatchNorm2d(features_d * 4),
            nn.LeakyReLU(0.2, inplace= True),
            nn.Conv2d(features_d * 4, features_d * 8, 4, 2, 1, bias= False),
            nn.BatchNorm2d(features_d * 8),
            nn.LeakyReLU(0.2, inplace= True),
            nn.Conv2d(features_d * 8, 1, 4, 1, 0, bias= False),
            nn.Sigmoid()
        )

    def forward(self, x, labels):
        embedding = self.embed(labels).view(-1, 1, self.img_size, self.img_size)
        x = torch.cat([x, embedding], dim= 1) 
        return self.net(x)

In [2]:
def load_pretrained_weights(model, path, target_classes, is_generator= True):
    state_dict = torch.load(path, map_location= DEVICE)
    embed_key = "embed.weight"
    
    if embed_key in state_dict:
        checkpoint_classes, embed_dim = state_dict[embed_key].shape
        if checkpoint_classes != target_classes:
            print(f"[!] Mismatch detected! Checkpoint has {checkpoint_classes} classes, target has {target_classes}.")
            print("--> Isolating and resetting embedding layers while maintaining structural convolutional priors...")
            del state_dict[embed_key]
            
            model.load_state_dict(state_dict, strict=False)
            if is_generator:
                model.embed = nn.Embedding(target_classes, embed_dim).to(DEVICE)
            else:
                model.embed = nn.Embedding(target_classes, IMAGE_SIZE * IMAGE_SIZE).to(DEVICE)
            return model
            
    model.load_state_dict(state_dict)
    return model

gen = Generator(DRONE_CLASSES, LATENT_DIM, EMBED_SIZE, CHANNELS_IMG, FEATURES_G).to(DEVICE)
disc = Discriminator(DRONE_CLASSES, CHANNELS_IMG, FEATURES_D, IMAGE_SIZE).to(DEVICE)

gen = load_pretrained_weights(gen, "models/pv_generator.pth", DRONE_CLASSES, is_generator=True)
disc = load_pretrained_weights(disc, "models/pv_discriminator.pth", DRONE_CLASSES, is_generator=False)
gen.train()
disc.train()

[!] Mismatch detected! Checkpoint has 38 classes, target has 11.
--> Isolating and resetting embedding layers while maintaining structural convolutional priors...
[!] Mismatch detected! Checkpoint has 38 classes, target has 11.
--> Isolating and resetting embedding layers while maintaining structural convolutional priors...


Discriminator(
  (embed): Embedding(11, 4096)
  (net): Sequential(
    (0): Conv2d(4, 64, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), bias=False)
    (1): LeakyReLU(negative_slope=0.2, inplace=True)
    (2): Conv2d(64, 128, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), bias=False)
    (3): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (4): LeakyReLU(negative_slope=0.2, inplace=True)
    (5): Conv2d(128, 256, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), bias=False)
    (6): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (7): LeakyReLU(negative_slope=0.2, inplace=True)
    (8): Conv2d(256, 512, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), bias=False)
    (9): BatchNorm2d(512, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (10): LeakyReLU(negative_slope=0.2, inplace=True)
    (11): Conv2d(512, 1, kernel_size=(4, 4), stride=(1, 1), bias=

In [5]:
transform = transforms.Compose([transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)), transforms.RandomHorizontalFlip(p=0.5),
                                transforms.ToTensor(), transforms.Normalize([0.5]*CHANNELS_IMG, [0.5]*CHANNELS_IMG)])

DRONE_DATA_PATH = "process_video/process_video/dataset/crops" 
dataset = datasets.ImageFolder(root=DRONE_DATA_PATH, transform=transform)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

opt_gen = optim.Adam(gen.parameters(), lr=FINE_TUNE_LR, betas=(0.5, 0.999))
opt_disc = optim.Adam(disc.parameters(), lr=FINE_TUNE_LR, betas=(0.5, 0.999))
criterion = nn.BCELoss()

print(dataset.classes)
print(len(dataset.classes))

print("Fine-tuning generator on drone imagery distribution...")
for epoch in range(FT_EPOCHS):
    for batch_idx, (real, labels) in enumerate(loader):
        real = real.to(DEVICE)
        labels = labels.to(DEVICE)
        b_size = real.shape[0]

        noise = torch.randn(b_size, LATENT_DIM, 1, 1).to(DEVICE)
        fake = gen(noise, labels)
        disc_real = disc(real, labels).view(-1)
        loss_D_real = criterion(disc_real, torch.ones_like(disc_real))
        disc_fake = disc(fake.detach(), labels).view(-1)
        loss_D_fake = criterion(disc_fake, torch.zeros_like(disc_fake))
        loss_D = (loss_D_real + loss_D_fake) / 2
        disc.zero_grad()
        loss_D.backward()
        opt_disc.step()

        output = disc(fake, labels).view(-1)
        loss_G = criterion(output, torch.ones_like(output))
        gen.zero_grad()
        loss_G.backward()
        opt_gen.step()

    if (epoch + 1) % 5 == 0 or epoch == FT_EPOCHS - 1:
        print(f"Fine-tune Epoch [{epoch+1}/{FT_EPOCHS}] | Loss D: {loss_D:.4f}, Loss G: {loss_G:.4f}")

torch.save(gen.state_dict(), "models/drone_generator.pth")
torch.save(disc.state_dict(), "models/drone_discriminator.pth")
print("Fine-tuning completed. Drone-adapted generator saved successfully!")

['Apple___Black_rot', 'Corn_(maize)___Common_rust_', 'Corn_(maize)___Northern_Leaf_Blight', 'Potato___Early_blight', 'Squash___Powdery_mildew', 'Strawberry___Leaf_scorch', 'Tomato___Bacterial_spot', 'Tomato___Early_blight', 'Tomato___Late_blight', 'Tomato___Septoria_leaf_spot', 'Tomato___Tomato_Yellow_Leaf_Curl_Virus']
11
Fine-tuning generator on drone imagery distribution...
Fine-tune Epoch [5/30] | Loss D: 0.3701, Loss G: 2.4596
Fine-tune Epoch [10/30] | Loss D: 0.2607, Loss G: 2.5464
Fine-tune Epoch [15/30] | Loss D: 0.2404, Loss G: 3.3482
Fine-tune Epoch [20/30] | Loss D: 0.2025, Loss G: 3.7328
Fine-tune Epoch [25/30] | Loss D: 0.1338, Loss G: 2.8919
Fine-tune Epoch [30/30] | Loss D: 0.1663, Loss G: 3.7143
Fine-tuning completed. Drone-adapted generator saved successfully!
